# KhaṭṭVision — Full DuwatBench Fine-tuning Run

**Goal:** fine-tune Muse Glimmer 30B to transcribe Arabic calligraphy, identify its canonical style and theme, and return normalized text-region boxes.

This Kaggle-ready full run includes:

- 4-bit Muse Glimmer 30B with LoRA over vision and language layers.
- A leakage-aware 70/15/15 split grouped by normalized transcription, source URL, and perceptual-image near duplicates.
- Four training tasks: full OCR, structured JSON, canonical theme classification, and cropped-region OCR. Style supervision remains inside every structured example.
- A closed nine-label theme taxonomy embedded directly in the instructions.
- Base and tuned OCR evaluation on the same randomized 50-image held-out subset.
- CER, WER, chrF2, exact match, JSON validity, style/theme accuracy, and localization metrics.
- Periodic checkpoints, prediction files, metrics, adapter export, and optional Hugging Face upload.

## Before running

1. Enable **Internet** and select **GPU T4 x2** in Kaggle.
2. Add a Kaggle secret named `HF_TOKEN` if Hugging Face authentication is required.
3. Restart the Kaggle session so no smoke-run adapter remains in memory.
4. Run all cells in order.
5. Do not resume from the earlier smoke checkpoint; it used the old unconstrained prompt.

### Expected runtime

The full configuration uses all 889 training images for one epoch. Based on the smoke run, expect roughly 9–11 hours on dual T4. The two GPUs primarily hold model shards; they do not provide ordinary two-worker data parallelism.

### Scientific-use note

DuwatBench exposes one public split. Compare models only on the held-out indices generated here. Do not compare this fine-tuned split directly with published zero-shot scores over the complete dataset.


In [ ]:
# Install before importing torch or transformers.
!pip install -q pip3-autoremove
!pip install -q torch torchvision torchaudio xformers --index-url https://download.pytorch.org/whl/cu128
!pip install -q unsloth
!pip install -q --no-deps --upgrade "torchao>=0.16.0"
!pip install -q "transformers==5.15.0" --no-deps
!pip install -q "trl==0.22.2" --no-deps
!pip install -q datasets accelerate peft bitsandbytes scikit-learn ImageHash jiwer sacrebleu pandas matplotlib tqdm huggingface_hub

## 1. Full-run configuration

This notebook is already configured for the complete training split and one epoch on Kaggle dual T4. The 448×448 pixel-area cap is retained for VRAM stability.


In [ ]:
import hashlib
import json
import math
import os
import random
import re
import unicodedata
from collections import Counter
from pathlib import Path

os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

import numpy as np
import pandas as pd
import torch
from PIL import Image

SEED = 3407
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

RUN_MODE = "full"
MODEL_NAME = "unsloth/Muse-Glimmer-30B-unsloth-bnb-4bit"
BF16_BASE_NAME = "unsloth/Muse-Glimmer-30B"
DATASET_NAME = "MBZUAI/DuwatBench"

MAX_SEQ_LENGTH = 1024
MAX_IMAGE_PIXELS = 448 * 448
MAX_IMAGE_SIDE = 896

LEARNING_RATE = 5e-5
NUM_EPOCHS = 1
LORA_R = 16
LORA_ALPHA = 16
GRAD_ACCUM = 8

# Retained only so the helper cells remain reusable.
SMOKE_TRAIN_IMAGES = 80
SMOKE_VAL_IMAGES = 20
SMOKE_MAX_STEPS = 15

RUN_BASELINE_EVAL = True
EVAL_LIMIT = 50
BASELINE_LIMIT = EVAL_LIMIT

MAX_NEW_TOKENS_OCR = 192
MAX_NEW_TOKENS_STRUCTURED = 512

# Set this only when resuming a checkpoint produced by THIS full notebook.
# Never point it at the old smoke checkpoint.
RESUME_FROM_CHECKPOINT = None

WORK_DIR = Path("/kaggle/working/khattvision_full")
OUTPUT_DIR = WORK_DIR / "checkpoints"
ADAPTER_DIR = WORK_DIR / "khattvision_lora"
WORK_DIR.mkdir(parents=True, exist_ok=True)

print("Run mode:", RUN_MODE)
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("Visible GPUs:", torch.cuda.device_count())
for i in range(torch.cuda.device_count()):
    p = torch.cuda.get_device_properties(i)
    print(f"  GPU {i}: {p.name} — {p.total_memory / 2**30:.1f} GiB")

In [ ]:
# Optional Hugging Face authentication through Kaggle Secrets.
from huggingface_hub import login

HF_TOKEN = ""
try:
    from kaggle_secrets import UserSecretsClient

    HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
except Exception as exc:
    print("HF_TOKEN was not loaded from Kaggle Secrets:", type(exc).__name__)

if HF_TOKEN:
    login(token=HF_TOKEN, add_to_git_credential=False)
    print("Authenticated with Hugging Face.")
else:
    print("Continuing without a token. Add the HF_TOKEN secret if model loading is denied.")

## 2. Load Muse Glimmer and add LoRA

Embedding offload is enabled only when one GPU is visible. With two 16 GB GPUs the model must remain shardable, so embedding offload is disabled.

In [ ]:
import transformers
from transformers import AutoConfig
from unsloth import FastModel

print("Transformers:", transformers.__version__)
assert transformers.__version__ == "5.15.0", transformers.__version__

config = AutoConfig.from_pretrained(MODEL_NAME, token=HF_TOKEN)
print("Architecture:", config.model_type)

OFFLOAD_EMBEDDING = torch.cuda.device_count() == 1
print("offload_embedding:", OFFLOAD_EMBEDDING)

model, processor = FastModel.from_pretrained(
    model_name=MODEL_NAME,
    load_in_4bit=True,
    max_seq_length=MAX_SEQ_LENGTH,
    use_gradient_checkpointing="unsloth",
    offload_embedding=OFFLOAD_EMBEDDING,
    token=HF_TOKEN,
)

print(type(model).__name__, type(processor).__name__)
print("image token:", processor.image_token, processor.image_token_id)

# Avoid the torchvision<0.27 LANCZOS fallback warning and use the same
# interpolation consistently in training and inference.
try:
    processor.image_processor.resample = Image.Resampling.BICUBIC
except Exception:
    pass

In [ ]:
model = FastModel.get_peft_model(
    model,
    finetune_vision_layers=True,
    finetune_language_layers=True,
    finetune_attention_modules=True,
    finetune_mlp_modules=True,
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=0,
    bias="none",
    random_state=SEED,
    use_rslora=False,
    loftq_config=None,
)
model.print_trainable_parameters()

## 3. Load and audit DuwatBench

The raw `style` field contains canonical single styles as well as comma-separated combinations. We canonicalize these into a six-label set without changing the original value.

In [ ]:
from datasets import load_dataset

raw_dataset = load_dataset(DATASET_NAME, split="train")
print(raw_dataset)
print(raw_dataset.features)

CANONICAL_STYLES = ["Thuluth", "Diwani", "Naskh", "Kufic", "Ruq'ah", "Nasta'liq"]


def parse_styles(value):
    styles = [s.strip() for s in re.split(r"\s*,\s*", value.strip()) if s.strip()]
    unknown = sorted(set(styles) - set(CANONICAL_STYLES))
    if unknown:
        raise ValueError(f"Unknown style labels: {unknown}")
    return sorted(set(styles), key=CANONICAL_STYLES.index)


style_counts = Counter(s for row in raw_dataset for s in parse_styles(row["style"]))
category_counts = Counter(raw_dataset["category"])
region_count = sum(len(x) for x in raw_dataset["text"])
word_count = sum(raw_dataset["total_words"])

print("Rows:", len(raw_dataset))
print("Annotated regions:", region_count)
print("Total words:", word_count)
print("Styles:", style_counts)
print("Categories:", category_counts)

In [ ]:
# Inspect one sample and confirm the annotation geometry.
import matplotlib.patches as patches
import matplotlib.pyplot as plt

sample = raw_dataset[0]
image = sample["image"].convert("RGB")
fig, ax = plt.subplots(figsize=(10, 8))
ax.imshow(image)
for bbox, text in zip(sample["bboxes"], sample["text"], strict=False):
    x, y, w, h = bbox  # DuwatBench uses xywh.
    ax.add_patch(patches.Rectangle((x, y), w, h, fill=False, edgecolor="red", linewidth=2))
    ax.text(x, max(0, y - 5), text[:35], color="red", fontsize=9, backgroundcolor="white")
ax.set_title(f"{sample['image_id']} | {sample['style']} | {sample['category']}")
ax.axis("off")
plt.show()
print(sample["text"])

## 4. Leakage-aware split

A random split is unsafe because common Arabic phrases occur in multiple images. The grouping below joins samples that share:

- the same lightly normalized full transcription;
- the same non-empty HTTP source URL; or
- a perceptual hash within Hamming distance 4.

We then search multiple group-level splits and select the one with the closest style and theme distributions. This prevents exact content and near-image overlap while keeping the held-out sets useful.

Normalization is used **only for grouping and normalized evaluation**. Original Arabic remains the training target.

In [ ]:
import imagehash
from sklearn.model_selection import GroupShuffleSplit
from tqdm.auto import tqdm

ARABIC_DIACRITICS = re.compile("[\u064b-\u065f\u0670\u06d6-\u06ed]")


def normalize_arabic(text):
    text = unicodedata.normalize("NFKC", str(text))
    text = ARABIC_DIACRITICS.sub("", text)
    text = re.sub("[إأآٱ]", "ا", text)
    text = text.replace("ى", "ي").replace("ة", "ه")
    text = re.sub(r"[^\u0600-\u06FF0-9\s]", " ", text)
    return re.sub(r"\s+", " ", text).strip()


def document_text_key(row):
    # Sorting makes the key insensitive to annotation-region order.
    regions = sorted(normalize_arabic(t) for t in row["text"])
    return " || ".join(regions)


class UnionFind:
    def __init__(self, n):
        self.parent = list(range(n))
        self.rank = [0] * n

    def find(self, x):
        while self.parent[x] != x:
            self.parent[x] = self.parent[self.parent[x]]
            x = self.parent[x]
        return x

    def union(self, a, b):
        ra, rb = self.find(a), self.find(b)
        if ra == rb:
            return
        if self.rank[ra] < self.rank[rb]:
            ra, rb = rb, ra
        self.parent[rb] = ra
        if self.rank[ra] == self.rank[rb]:
            self.rank[ra] += 1


n = len(raw_dataset)
uf = UnionFind(n)

# Group identical normalized full transcriptions.
first_by_text = {}
text_keys = []
for i in range(n):
    key = document_text_key(raw_dataset[i])
    text_keys.append(key)
    if key in first_by_text:
        uf.union(i, first_by_text[key])
    else:
        first_by_text[key] = i

# Group identical non-empty source URLs.
first_by_source = {}
for i in range(n):
    source = str(raw_dataset[i].get("source") or "").strip()
    if source.startswith(("http://", "https://")):
        if source in first_by_source:
            uf.union(i, first_by_source[source])
        else:
            first_by_source[source] = i

# Perceptual near-duplicate grouping. 1,272 images means ~0.8M cheap hash comparisons.
phashes = []
for i in tqdm(range(n), desc="Computing perceptual hashes"):
    img = raw_dataset[i]["image"].convert("RGB")
    phashes.append(imagehash.phash(img, hash_size=8))

PHASH_DISTANCE = 4
for i in tqdm(range(n), desc="Grouping near-duplicate images"):
    for j in range(i + 1, n):
        if phashes[i] - phashes[j] <= PHASH_DISTANCE:
            uf.union(i, j)

groups = np.array([uf.find(i) for i in range(n)])
print("Normalized full-text keys:", len(set(text_keys)))
print("Combined split groups:", len(set(groups)))
print("Largest group size:", Counter(groups).most_common(1)[0][1])

In [ ]:
# Pick the best of several reproducible group splits by distribution balance.
style_matrix = np.zeros((n, len(CANONICAL_STYLES)), dtype=np.float32)
themes = sorted(set(raw_dataset["category"]))
theme_matrix = np.zeros((n, len(themes)), dtype=np.float32)

for i in range(n):
    for style in parse_styles(raw_dataset[i]["style"]):
        style_matrix[i, CANONICAL_STYLES.index(style)] = 1
    theme_matrix[i, themes.index(raw_dataset[i]["category"])] = 1


def distribution(arr, idx):
    values = arr[idx].sum(axis=0)
    return values / max(values.sum(), 1)


global_style = distribution(style_matrix, np.arange(n))
global_theme = distribution(theme_matrix, np.arange(n))


def allocation_error(matrix, idx, target_ratio):
    total_per_label = matrix.sum(axis=0)
    split_per_label = matrix[idx].sum(axis=0)
    observed_ratio = split_per_label / np.maximum(total_per_label, 1)
    return float(np.abs(observed_ratio - target_ratio).mean())


def has_required_coverage(train_idx, val_idx, test_idx):
    train_theme = theme_matrix[train_idx].sum(axis=0)
    val_theme = theme_matrix[val_idx].sum(axis=0)
    test_theme = theme_matrix[test_idx].sum(axis=0)
    train_style = style_matrix[train_idx].sum(axis=0)
    val_style = style_matrix[val_idx].sum(axis=0)
    test_style = style_matrix[test_idx].sum(axis=0)
    return (
        np.all(train_theme >= 2)
        and np.all(val_theme >= 1)
        and np.all(test_theme >= 1)
        and np.all(train_style >= 2)
        and np.all(val_style >= 1)
        and np.all(test_style >= 1)
    )


def split_score(train_idx, val_idx, test_idx):
    if not has_required_coverage(train_idx, val_idx, test_idx):
        return float("inf")
    score = 0.0
    for idx, target_size in [(train_idx, 0.70), (val_idx, 0.15), (test_idx, 0.15)]:
        score += abs(len(idx) / n - target_size) * 4
        score += np.abs(distribution(style_matrix, idx) - global_style).mean()
        score += np.abs(distribution(theme_matrix, idx) - global_theme).mean()
        score += allocation_error(style_matrix, idx, target_size)
        score += allocation_error(theme_matrix, idx, target_size)
    return float(score)


candidates = []
all_idx = np.arange(n)
for seed in range(SEED, SEED + 5000):
    outer = GroupShuffleSplit(n_splits=1, train_size=0.70, random_state=seed)
    train_pos, rest_pos = next(outer.split(all_idx, groups=groups))
    train_idx = all_idx[train_pos]
    rest_idx = all_idx[rest_pos]
    rest_groups = groups[rest_pos]

    inner = GroupShuffleSplit(n_splits=1, train_size=0.50, random_state=seed + 10_000)
    val_pos, test_pos = next(inner.split(rest_idx, groups=rest_groups))
    val_idx = rest_idx[val_pos]
    test_idx = rest_idx[test_pos]
    candidates.append((split_score(train_idx, val_idx, test_idx), seed, train_idx, val_idx, test_idx))

valid_candidates = [candidate for candidate in candidates if np.isfinite(candidate[0])]
assert valid_candidates, (
    "No split satisfied category/style coverage. Inspect rare-label groups " "before relaxing the minimum counts."
)
best_score, split_seed, train_idx, val_idx, test_idx = min(valid_candidates, key=lambda x: x[0])
print("Selected seed:", split_seed, "balance score:", round(best_score, 5))
print("Sizes:", {"train": len(train_idx), "validation": len(val_idx), "test": len(test_idx)})


def assert_disjoint(a, b, label):
    overlap = set(groups[a]) & set(groups[b])
    assert not overlap, f"Group leakage between {label}: {len(overlap)} groups"


assert_disjoint(train_idx, val_idx, "train/validation")
assert_disjoint(train_idx, test_idx, "train/test")
assert_disjoint(val_idx, test_idx, "validation/test")

split_name = {}
for idx in train_idx:
    split_name[int(idx)] = "train"
for idx in val_idx:
    split_name[int(idx)] = "validation"
for idx in test_idx:
    split_name[int(idx)] = "test"

manifest = pd.DataFrame(
    {
        "row_index": np.arange(n),
        "image_id": raw_dataset["image_id"],
        "split": [split_name[i] for i in range(n)],
        "group": groups,
        "normalized_full_text_sha1": [hashlib.sha1(k.encode("utf-8")).hexdigest() for k in text_keys],
        "style": raw_dataset["style"],
        "category": raw_dataset["category"],
    }
)
manifest.to_csv(WORK_DIR / "duwatbench_split_manifest.csv", index=False)

canonical_style_rows = []
for i in range(n):
    for style in parse_styles(raw_dataset[i]["style"]):
        canonical_style_rows.append({"split": split_name[i], "style": style})
canonical_style_df = pd.DataFrame(canonical_style_rows)
canonical_style_table = pd.crosstab(canonical_style_df["split"], canonical_style_df["style"]).reindex(
    index=["train", "validation", "test"], columns=CANONICAL_STYLES, fill_value=0
)
category_table = pd.crosstab(manifest["split"], manifest["category"]).reindex(
    index=["train", "validation", "test"], columns=themes, fill_value=0
)
display(canonical_style_table)
display(category_table)
print("Split validation passed; all canonical styles and themes are represented.")

In [ ]:
# Use the same randomized held-out rows for base and tuned evaluation.
eval_rng = np.random.default_rng(SEED + 99)
eval_idx = eval_rng.permutation(np.asarray(test_idx, dtype=int))[:EVAL_LIMIT]

assert len(eval_idx) == EVAL_LIMIT
assert set(eval_idx).issubset(set(test_idx))

eval_manifest = manifest[manifest["row_index"].isin(eval_idx)].copy()
print("Evaluation rows:", eval_idx.tolist())
display(pd.crosstab(eval_manifest["category"], columns="count"))
display(pd.crosstab(eval_manifest["style"], columns="count"))

## 5. Build the four full-run instruction tasks

The full run trains structured JSON, full-image OCR, cropped-region OCR, and explicit closed-taxonomy theme classification. Style remains supervised inside every structured target.

Bounding boxes are converted from pixel `xywh` to normalized `xywh` coordinates in `[0, 1000]`. They are normalized against the original image geometry before the image is resized.


In [ ]:
THEME_LABELS = [
    "dedication",
    "devotional invocation",
    "hadith",
    "names of Allah",
    "names of companions",
    "names of the Prophet",
    "non-religious",
    "personal/place name",
    "quranic",
]

THEME_OPTIONS = json.dumps(THEME_LABELS, ensure_ascii=False)
STYLE_OPTIONS = json.dumps(CANONICAL_STYLES, ensure_ascii=False)

PROMPTS = {
    "full_ocr": ("اقرأ جميع النصوص العربية الظاهرة في لوحة الخط. " "أعد النص فقط دون شرح، وافصل المقاطع بسطر جديد."),
    "crop_ocr": ("انسخ النص العربي الظاهر في هذا المقطع بدقة. " "أعد النص فقط دون شرح."),
    "theme": f"""
صنّف موضوع لوحة الخط العربي.

اختر قيمة واحدة فقط من القائمة التالية، واكتبها حرفيًا:
{THEME_OPTIONS}

ممنوع استخدام أي قيمة أخرى أو مرادف، بما في ذلك:
religious, Islamic, Quranic, Qur'an, Dua, personal name.

أعد التصنيف فقط دون شرح أو علامات ترقيم.
""".strip(),
    "style": f"""
حدّد نوع أو أنواع الخط العربي في الصورة.

استخدم فقط القيم التالية وبنفس الكتابة:
{STYLE_OPTIONS}

أعد مصفوفة JSON صالحة فقط، مثل:
["Diwani"]

لا تضف شرحًا أو Markdown.
""".strip(),
    "structured": f"""
حلّل لوحة الخط العربي وحدّد أنواع الخط، والتصنيف الموضوعي،
والنصوص العربية ومواقعها.

القيم المسموح بها في styles فقط:
{STYLE_OPTIONS}

يجب أن تكون theme قيمة واحدة فقط من:
{THEME_OPTIONS}

تعليمات إلزامية:
- استخدم أسماء التصنيفات السابقة حرفيًا.
- لا تستخدم مرادفات مثل religious أو Islamic أو Dua أو Quranic.
- استخدم bbox_1000_xywh بصيغة [x, y, width, height].
- يجب أن تكون جميع الإحداثيات أعدادًا صحيحة بين 0 و1000.
- أعد JSON صالحًا فقط.
- لا تضف شرحًا أو Markdown.

البنية المطلوبة:
{{
  "styles": ["Diwani"],
  "theme": "quranic",
  "regions": [
    {{
      "bbox_1000_xywh": [100, 100, 700, 300],
      "text": "النص العربي"
    }}
  ]
}}
""".strip(),
}


def prepare_image_for_model(image):
    # Resize without changing aspect ratio. The pixel-area cap controls
    # visual patch tokens; dimensions are rounded to the merged 28px grid.
    image = image.convert("RGB")
    width, height = image.size
    area_scale = math.sqrt(MAX_IMAGE_PIXELS / max(width * height, 1))
    side_scale = MAX_IMAGE_SIDE / max(width, height)
    scale = min(1.0, area_scale, side_scale)
    if scale >= 1.0:
        return image
    new_width = max(28, int((width * scale) // 28) * 28)
    new_height = max(28, int((height * scale) // 28) * 28)
    return image.resize((new_width, new_height), Image.Resampling.BICUBIC)


def normalize_bbox_xywh(box, width, height):
    x, y, w, h = (float(v) for v in box)
    return [
        int(round(1000 * x / width)),
        int(round(1000 * y / height)),
        int(round(1000 * w / width)),
        int(round(1000 * h / height)),
    ]


def safe_crop(image, box, pad=4):
    x, y, w, h = (int(round(v)) for v in box)
    left = max(0, x - pad)
    top = max(0, y - pad)
    right = min(image.width, x + w + pad)
    bottom = min(image.height, y + h + pad)
    return image.crop((left, top, right, bottom)).convert("RGB")


def conversation(image, instruction, answer):
    return {
        "messages": [
            {
                "role": "user",
                "content": [
                    {"type": "image", "image": image},
                    {"type": "text", "text": instruction},
                ],
            },
            {
                "role": "assistant",
                "content": [{"type": "text", "text": answer}],
            },
        ]
    }


def structured_target(sample, image):
    regions = [
        {
            "bbox_1000_xywh": normalize_bbox_xywh(box, image.width, image.height),
            "text": text,
        }
        for box, text in zip(sample["bboxes"], sample["text"], strict=False)
    ]
    return {
        "styles": parse_styles(sample["style"]),
        "theme": sample["category"],
        "regions": regions,
    }


def build_examples(indices, tasks):
    examples = []
    task_counts = Counter()
    for idx in tqdm([int(i) for i in indices], desc="Building examples"):
        sample = raw_dataset[idx]
        source_image = sample["image"].convert("RGB")
        image = prepare_image_for_model(source_image)

        if "full_ocr" in tasks:
            examples.append(conversation(image, PROMPTS["full_ocr"], "\n".join(sample["text"])))
            task_counts["full_ocr"] += 1

        if "structured" in tasks:
            # Normalize boxes against original geometry, not resized pixels.
            answer = json.dumps(structured_target(sample, source_image), ensure_ascii=False, separators=(",", ":"))
            examples.append(conversation(image, PROMPTS["structured"], answer))
            task_counts["structured"] += 1

        if "style" in tasks:
            answer = json.dumps(parse_styles(sample["style"]), ensure_ascii=False)
            examples.append(conversation(image, PROMPTS["style"], answer))
            task_counts["style"] += 1

        if "theme" in tasks:
            examples.append(conversation(image, PROMPTS["theme"], sample["category"]))
            task_counts["theme"] += 1

        if "crop_ocr" in tasks:
            for box, text in zip(sample["bboxes"], sample["text"], strict=False):
                crop = prepare_image_for_model(safe_crop(source_image, box))
                examples.append(conversation(crop, PROMPTS["crop_ocr"], text))
                task_counts["crop_ocr"] += 1

    random.Random(SEED).shuffle(examples)
    return examples, task_counts


if RUN_MODE == "smoke":
    active_train_idx = train_idx[:SMOKE_TRAIN_IMAGES]
    active_val_idx = val_idx[:SMOKE_VAL_IMAGES]
    train_tasks = ["structured", "full_ocr", "crop_ocr"]
else:
    active_train_idx = train_idx
    active_val_idx = val_idx
    train_tasks = ["structured", "full_ocr", "crop_ocr", "theme"]

train_examples, train_task_counts = build_examples(active_train_idx, train_tasks)
# Validation loss uses the two main generation tasks and stays reasonably fast.
val_examples, val_task_counts = build_examples(active_val_idx, ["structured", "full_ocr"])

print("Train examples:", len(train_examples), train_task_counts)
print("Validation examples:", len(val_examples), val_task_counts)

if RUN_MODE == "full":
    assert train_task_counts["structured"] == len(train_idx)
    assert train_task_counts["full_ocr"] == len(train_idx)
    assert train_task_counts["theme"] == len(train_idx)
    assert train_task_counts["crop_ocr"] > len(train_idx)
    assert val_task_counts["structured"] == len(val_idx)
    assert val_task_counts["full_ocr"] == len(val_idx)
    print("Full-run task coverage checks passed.")

In [ ]:
# Inspect the exact serialized ATEM template and one target.
preview = train_examples[0]
rendered = processor.apply_chat_template(
    preview["messages"],
    tokenize=False,
    add_generation_prompt=False,
)
print(rendered[:4000])

## 6. Generation and evaluation utilities

These functions are defined before training so the initialized LoRA model can be evaluated as the base-model baseline. LoRA starts at a zero update, so its initial output should match the base checkpoint up to numerical differences.

In [ ]:
from jiwer import cer as jiwer_cer
from jiwer import wer as jiwer_wer
from sacrebleu.metrics import CHRF

chrf_metric = CHRF(word_order=2)


def muse_glimmer_prompt(messages, answer_directly=True):
    text = processor.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=False,
    )
    return text + " to=user<|message|>" if answer_directly else text


@torch.inference_mode()
def generate_answer(image, instruction, max_new_tokens=384):
    model.eval()
    image = prepare_image_for_model(image)
    messages = [
        {
            "role": "user",
            "content": [
                {"type": "image"},
                {"type": "text", "text": instruction},
            ],
        }
    ]
    inputs = processor(
        text=[muse_glimmer_prompt(messages)],
        images=[[image]],
        add_special_tokens=False,
        return_tensors="pt",
    ).to("cuda")

    output = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        use_cache=True,
        do_sample=False,
    )
    new_tokens = output[:, inputs["input_ids"].shape[1] :]
    text = processor.tokenizer.batch_decode(new_tokens, skip_special_tokens=True)[0].strip()
    del inputs, output, new_tokens
    return text


def safe_metric_text(text):
    text = normalize_arabic(text)
    return text if text else "∅"


def aggregate_ocr_metrics(records):
    refs = [safe_metric_text(r["reference"]) for r in records]
    preds = [safe_metric_text(r["prediction"]) for r in records]
    return {
        "samples": len(records),
        "cer_normalized": float(jiwer_cer(refs, preds)),
        "wer_normalized": float(jiwer_wer(refs, preds)),
        "chrf2_normalized": float(chrf_metric.corpus_score(preds, [refs]).score),
        "exact_match_normalized": float(np.mean([p == r for p, r in zip(preds, refs, strict=False)])),
    }


def evaluate_ocr(indices, limit=None, prediction_path=None):
    selected = [int(i) for i in indices]
    if limit is not None:
        selected = selected[:limit]
    records = []
    for idx in tqdm(selected, desc="OCR evaluation"):
        sample = raw_dataset[idx]
        reference = "\n".join(sample["text"])
        prediction = generate_answer(sample["image"], PROMPTS["full_ocr"], MAX_NEW_TOKENS_OCR)
        records.append(
            {
                "row_index": idx,
                "image_id": sample["image_id"],
                "reference": reference,
                "prediction": prediction,
                "style": sample["style"],
                "category": sample["category"],
            }
        )
    if prediction_path:
        with open(prediction_path, "w", encoding="utf-8") as f:
            for row in records:
                f.write(json.dumps(row, ensure_ascii=False) + "\n")
    return aggregate_ocr_metrics(records), records


def extract_json(text):
    start, end = text.find("{"), text.rfind("}")
    if start < 0 or end < start:
        return None
    try:
        return json.loads(text[start : end + 1])
    except Exception:
        return None


def bbox_iou_xywh(a, b):
    ax, ay, aw, ah = (float(v) for v in a)
    bx, by, bw, bh = (float(v) for v in b)
    ax2, ay2, bx2, by2 = ax + aw, ay + ah, bx + bw, by + bh
    ix1, iy1 = max(ax, bx), max(ay, by)
    ix2, iy2 = min(ax2, bx2), min(ay2, by2)
    inter = max(0.0, ix2 - ix1) * max(0.0, iy2 - iy1)
    union = aw * ah + bw * bh - inter
    return inter / union if union > 0 else 0.0


def greedy_region_matches(gold_regions, pred_regions):
    candidates = []
    for gi, gold in enumerate(gold_regions):
        for pi, pred in enumerate(pred_regions):
            try:
                iou = bbox_iou_xywh(gold["bbox_1000_xywh"], pred["bbox_1000_xywh"])
            except Exception:
                iou = 0.0
            candidates.append((iou, gi, pi))
    used_g, used_p, matches = set(), set(), []
    for iou, gi, pi in sorted(candidates, reverse=True):
        if gi not in used_g and pi not in used_p:
            used_g.add(gi)
            used_p.add(pi)
            matches.append((iou, gi, pi))
    return matches


def evaluate_structured(indices, limit=None, prediction_path=None):
    selected = [int(i) for i in indices]
    if limit is not None:
        selected = selected[:limit]
    records, valid = [], []
    ious, matched_refs, matched_preds = [], [], []
    style_exact, theme_exact, gold_region_total, matched_at_50 = [], [], 0, 0

    for idx in tqdm(selected, desc="Structured evaluation"):
        sample = raw_dataset[idx]
        image = sample["image"].convert("RGB")
        gold = structured_target(sample, image)
        raw_prediction = generate_answer(image, PROMPTS["structured"], MAX_NEW_TOKENS_STRUCTURED)
        pred = extract_json(raw_prediction)
        is_valid = isinstance(pred, dict)
        valid.append(is_valid)
        row = {
            "row_index": idx,
            "image_id": sample["image_id"],
            "gold": gold,
            "raw_prediction": raw_prediction,
            "json_valid": is_valid,
        }

        if is_valid:
            pred_styles = pred.get("styles", [])
            if isinstance(pred_styles, str):
                pred_styles = [pred_styles]
            style_exact.append(set(pred_styles) == set(gold["styles"]))
            theme_exact.append(str(pred.get("theme", "")).strip() == gold["theme"])
            pred_regions = pred.get("regions", []) if isinstance(pred.get("regions", []), list) else []
            matches = greedy_region_matches(gold["regions"], pred_regions)
            gold_region_total += len(gold["regions"])
            for iou, gi, pi in matches:
                ious.append(iou)
                if iou >= 0.5:
                    matched_at_50 += 1
                    matched_refs.append(str(gold["regions"][gi].get("text", "")))
                    matched_preds.append(str(pred_regions[pi].get("text", "")))
        else:
            style_exact.append(False)
            theme_exact.append(False)
            gold_region_total += len(gold["regions"])
        records.append(row)

    region_cer = None
    if matched_refs:
        region_cer = float(
            jiwer_cer(
                [safe_metric_text(x) for x in matched_refs],
                [safe_metric_text(x) for x in matched_preds],
            )
        )
    metrics = {
        "samples": len(records),
        "json_valid_rate": float(np.mean(valid)) if valid else 0.0,
        "style_set_exact": float(np.mean(style_exact)) if style_exact else 0.0,
        "theme_accuracy": float(np.mean(theme_exact)) if theme_exact else 0.0,
        "mean_matched_iou": float(np.mean(ious)) if ious else 0.0,
        "box_recall_at_0.5": float(matched_at_50 / max(gold_region_total, 1)),
        "matched_region_cer_normalized": region_cer,
    }
    if prediction_path:
        with open(prediction_path, "w", encoding="utf-8") as f:
            for row in records:
                f.write(json.dumps(row, ensure_ascii=False) + "\n")
    return metrics, records

In [ ]:
# Base-model OCR evaluation on the exact rows used after fine-tuning.
baseline_metrics = {}
if RUN_BASELINE_EVAL:
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    with model.disable_adapter():
        baseline_ocr, baseline_records = evaluate_ocr(
            eval_idx,
            limit=None,
            prediction_path=WORK_DIR / "baseline_ocr_predictions.jsonl",
        )

    baseline_metrics["ocr"] = baseline_ocr
    print(json.dumps(baseline_metrics, indent=2, ensure_ascii=False))
    with open(WORK_DIR / "baseline_metrics.json", "w", encoding="utf-8") as f:
        json.dump(baseline_metrics, f, indent=2, ensure_ascii=False)

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

## 7. Trainer

The collator masks the image, instruction, and template tokens. Only the assistant answer contributes to the loss. The ATEM markers must remain exact.

`loss_type="nll"` is retained from the source notebook to avoid the gradient-accumulation scaling issue described for Muse Glimmer with chunked NLL.

In [ ]:
from trl import SFTConfig, SFTTrainer
from unsloth.trainer import UnslothVisionDataCollator

instruction_part = "<|start|>user<|message|>"
response_part = "<|start|>assistant to=user<|message|>"

collator = UnslothVisionDataCollator(
    model,
    processor,
    max_seq_length=MAX_SEQ_LENGTH,
    train_on_responses_only=True,
    instruction_part=instruction_part,
    response_part=response_part,
)

use_bf16 = bool(torch.cuda.is_available() and torch.cuda.is_bf16_supported())
print("bf16:", use_bf16, "fp16:", not use_bf16)

trainer_args = dict(
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=LEARNING_RATE,
    warmup_ratio=0.05,
    logging_steps=10,
    optim="adamw_8bit",
    weight_decay=0.001,
    lr_scheduler_type="cosine",
    seed=SEED,
    output_dir=str(OUTPUT_DIR),
    report_to="none",
    loss_type="nll",
    remove_unused_columns=False,
    dataset_text_field="",
    dataset_kwargs={"skip_prepare_dataset": True},
    max_length=MAX_SEQ_LENGTH,
    packing=False,
    eval_strategy="epoch",
    save_strategy="steps",
    save_steps=50,
    save_total_limit=2,
    load_best_model_at_end=False,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    bf16=use_bf16,
    fp16=not use_bf16,
    dataloader_num_workers=0,
)

if RUN_MODE == "smoke":
    trainer_args["max_steps"] = SMOKE_MAX_STEPS
else:
    trainer_args["num_train_epochs"] = NUM_EPOCHS

trainer = SFTTrainer(
    model=model,
    train_dataset=train_examples,
    eval_dataset=val_examples,
    processing_class=processor.tokenizer,
    data_collator=collator,
    args=SFTConfig(**trainer_args),
)

In [ ]:
# Verify response-only masking before spending GPU time.
batch = collator([train_examples[0], train_examples[1]])
labels = batch["labels"][0]
trained_ids = [token for token in labels.tolist() if token != -100]
print("Trained tokens:", len(trained_ids), "of", labels.numel())
print(processor.tokenizer.decode(trained_ids)[:3000])
assert trained_ids, "No assistant tokens were selected for training."

In [ ]:
# Train one complete epoch, or resume only from this full run's checkpoint.
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()

train_result = trainer.train(resume_from_checkpoint=RESUME_FROM_CHECKPOINT)
print(train_result.metrics)

if torch.cuda.is_available():
    print("Peak reserved GiB:", round(torch.cuda.max_memory_reserved() / 2**30, 3))

## 8. Held-out evaluation after full fine-tuning

The tuned OCR and structured model are evaluated on the same randomized 50 rows selected before the baseline. This makes the base-versus-tuned OCR comparison paired and directly interpretable.


In [ ]:
# Resume-safe final metrics assembly.
# This does NOT rerun OCR or structured evaluation.

baseline_path = WORK_DIR / "baseline_metrics.json"

if "baseline_metrics" not in globals():
    if baseline_path.exists():
        with open(baseline_path, encoding="utf-8") as f:
            baseline_metrics = json.load(f)
        print("Loaded baseline from:", baseline_path)
    else:
        # Previously measured on the same 50 evaluation rows.
        baseline_metrics = {
            "ocr": {
                "samples": 50,
                "cer_normalized": 0.7966887417218543,
                "wer_normalized": 1.1182432432432432,
                "chrf2_normalized": 41.47449532187935,
                "exact_match_normalized": 0.22,
            }
        }
        print("Using the previously measured baseline metrics.")

# Make sure the completed evaluations are still available.
assert "tuned_ocr_metrics" in globals(), "tuned_ocr_metrics is missing. Rerun only evaluate_ocr."
assert "tuned_structured_metrics" in globals(), "tuned_structured_metrics is missing. Rerun only evaluate_structured."
assert "train_result" in globals(), "train_result is missing."

validation_metrics = next(
    (row for row in reversed(trainer.state.log_history) if "eval_loss" in row),
    {},
)

final_metrics = {
    "run_mode": RUN_MODE,
    "split_seed": int(split_seed),
    "evaluation_rows": [int(i) for i in eval_idx],
    "baseline": baseline_metrics,
    "tuned_ocr": tuned_ocr_metrics,
    "tuned_structured": tuned_structured_metrics,
    "validation": validation_metrics,
    "train_metrics": train_result.metrics,
}

print(
    json.dumps(
        final_metrics,
        indent=2,
        ensure_ascii=False,
    )
)

with open(
    WORK_DIR / "final_metrics.json",
    "w",
    encoding="utf-8",
) as f:
    json.dump(
        final_metrics,
        f,
        indent=2,
        ensure_ascii=False,
    )

print("Saved:", WORK_DIR / "final_metrics.json")

In [ ]:
if torch.cuda.is_available():
    torch.cuda.empty_cache()

tuned_ocr_metrics, tuned_ocr_records = evaluate_ocr(
    eval_idx,
    limit=None,
    prediction_path=WORK_DIR / "tuned_ocr_predictions.jsonl",
)

if torch.cuda.is_available():
    torch.cuda.empty_cache()

tuned_structured_metrics, tuned_structured_records = evaluate_structured(
    eval_idx,
    limit=None,
    prediction_path=WORK_DIR / "tuned_structured_predictions.jsonl",
)

validation_metrics = next(
    (row for row in reversed(trainer.state.log_history) if "eval_loss" in row),
    {},
)

final_metrics = {
    "run_mode": RUN_MODE,
    "split_seed": int(split_seed),
    "evaluation_rows": [int(i) for i in eval_idx],
    "baseline": baseline_metrics,
    "tuned_ocr": tuned_ocr_metrics,
    "tuned_structured": tuned_structured_metrics,
    "validation": validation_metrics,
    "train_metrics": train_result.metrics,
}

print(json.dumps(final_metrics, indent=2, ensure_ascii=False))
with open(WORK_DIR / "final_metrics.json", "w", encoding="utf-8") as f:
    json.dump(final_metrics, f, indent=2, ensure_ascii=False)

In [ ]:
# Qualitative examples.
for row in tuned_ocr_records[:5]:
    print("=" * 100)
    print("Image:", row["image_id"], "| Style:", row["style"], "| Theme:", row["category"])
    print("REFERENCE:\n", row["reference"])
    print("PREDICTION:\n", row["prediction"])

## 9. Save and export

The default export saves only the LoRA adapters and processor. This is much smaller than a merged checkpoint and is the recommended Kaggle output.

In [ ]:
import shutil

model.save_pretrained(str(ADAPTER_DIR))
processor.save_pretrained(str(ADAPTER_DIR))

archive = shutil.make_archive(str(WORK_DIR / "khattvision_lora"), "zip", root_dir=ADAPTER_DIR)
print("Adapter directory:", ADAPTER_DIR)
print("Downloadable archive:", archive)
artifact_archive = shutil.make_archive(
    "/kaggle/working/khattvision_full_artifacts",
    "zip",
    root_dir=WORK_DIR,
)
print("All run artifacts:", WORK_DIR)
print("Downloadable artifact archive:", artifact_archive)

In [ ]:
# Optional: push adapters to your Hugging Face account.
PUSH_TO_HUB = True
HF_REPO = "Omartificial-Intelligence-Space/KhattVision-Muse-Glimmer-30B-LoRA"

HF_TOKEN = "<REDACTED: use HF_TOKEN from .env>"

if PUSH_TO_HUB:
    assert HF_TOKEN, "HF_TOKEN is required for upload."
    model.push_to_hub(HF_REPO, token=HF_TOKEN, private=True)
    processor.push_to_hub(HF_REPO, token=HF_TOKEN, private=True)
    print("Uploaded:", HF_REPO)

In [ ]:
# Optional: merge into a normal 16-bit checkpoint.
# This needs roughly 56 GB for the output plus download/staging space.
MERGE_16BIT = False

if MERGE_16BIT:
    model.config._name_or_path = BF16_BASE_NAME
    merged_dir = WORK_DIR / "khattvision_merged_16bit"
    model.save_pretrained_merged(str(merged_dir), processor)
    print("Merged checkpoint:", merged_dir)

## Final run checklist

- Confirm `RUN_MODE == "full"` and two T4 GPUs are visible.
- Confirm task counts include 889 structured, 889 full OCR, and 889 theme examples.
- Confirm response-only masking displays assistant targets only.
- Do not resume from the old smoke checkpoint.
- Preserve `duwatbench_split_manifest.csv`, `final_metrics.json`, and prediction JSONL files.
- Download both `khattvision_lora.zip` and `khattvision_full_artifacts.zip` after completion.
- For a paper-quality follow-up, repeat the full configuration with additional seeds and evaluate the complete held-out split.


In [ ]:
%pip install -q gradio

In [ ]:
import gc
import json
import re

import gradio as gr
import torch
from PIL import Image, ImageDraw

# Free training-only memory while keeping the trained model loaded.
if "trainer" in globals():
    try:
        trainer.optimizer = None
        trainer.lr_scheduler = None
    except Exception:
        pass

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

model.eval()


TASKS = {
    "Structured analysis": {
        "prompt": PROMPTS["structured"],
        "max_tokens": MAX_NEW_TOKENS_STRUCTURED,
    },
    "Full Arabic OCR": {
        "prompt": PROMPTS["full_ocr"],
        "max_tokens": MAX_NEW_TOKENS_OCR,
    },
    "Theme classification": {
        "prompt": PROMPTS["theme"],
        "max_tokens": 64,
    },
    "Calligraphy style": {
        "prompt": PROMPTS["style"],
        "max_tokens": 96,
    },
}


def parse_json_value(text):
    """Parse either a JSON object or JSON array from the output."""
    text = str(text).strip()

    # Remove optional Markdown fences.
    text = re.sub(r"^```(?:json)?\s*", "", text)
    text = re.sub(r"\s*```$", "", text)

    try:
        return json.loads(text)
    except Exception:
        pass

    candidates = []

    object_start = text.find("{")
    object_end = text.rfind("}")
    if object_start >= 0 and object_end > object_start:
        candidates.append(text[object_start : object_end + 1])

    array_start = text.find("[")
    array_end = text.rfind("]")
    if array_start >= 0 and array_end > array_start:
        candidates.append(text[array_start : array_end + 1])

    for candidate in candidates:
        try:
            return json.loads(candidate)
        except Exception:
            continue

    return None


def draw_predicted_regions(image, prediction):
    """Draw normalized bbox_1000_xywh predictions."""
    output = image.convert("RGB").copy()

    if not isinstance(prediction, dict):
        return output

    regions = prediction.get("regions", [])
    if not isinstance(regions, list):
        return output

    draw = ImageDraw.Draw(output)
    width, height = output.size

    colors = [
        "#FF2D55",
        "#007AFF",
        "#34C759",
        "#FF9500",
        "#AF52DE",
        "#00C7BE",
    ]

    for index, region in enumerate(regions):
        if not isinstance(region, dict):
            continue

        box = region.get("bbox_1000_xywh")
        if not isinstance(box, list) or len(box) != 4:
            continue

        try:
            x, y, w, h = (float(value) for value in box)
        except Exception:
            continue

        x1 = max(0, min(width, x * width / 1000))
        y1 = max(0, min(height, y * height / 1000))
        x2 = max(0, min(width, (x + w) * width / 1000))
        y2 = max(0, min(height, (y + h) * height / 1000))

        if x2 <= x1 or y2 <= y1:
            continue

        color = colors[index % len(colors)]

        draw.rectangle(
            [x1, y1, x2, y2],
            outline=color,
            width=max(3, int(min(width, height) / 180)),
        )

        # Numbering avoids Arabic font/shaping problems inside PIL.
        draw.rectangle(
            [x1, y1, x1 + 30, y1 + 25],
            fill=color,
        )
        draw.text(
            [x1 + 8, y1 + 5],
            str(index + 1),
            fill="white",
        )

    return output


def run_khattvision(uploaded_image, selected_task):
    if uploaded_image is None:
        raise gr.Error("Please upload an Arabic calligraphy image.")

    image = uploaded_image.convert("RGB")
    task_config = TASKS[selected_task]

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    try:
        raw_output = generate_answer(
            image=image,
            instruction=task_config["prompt"],
            max_new_tokens=task_config["max_tokens"],
        )
    except torch.cuda.OutOfMemoryError:
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        raise gr.Error("GPU memory is full. Wait for the current request to finish " "or upload a smaller image.")

    parsed = parse_json_value(raw_output)

    if selected_task == "Structured analysis":
        json_output = parsed if isinstance(parsed, dict) else {"parse_error": True, "raw_output": raw_output}
        annotated_image = draw_predicted_regions(image, parsed)

    elif selected_task == "Calligraphy style":
        json_output = {"styles": parsed if isinstance(parsed, list) else raw_output}
        annotated_image = image

    elif selected_task == "Theme classification":
        json_output = {"theme": raw_output.strip()}
        annotated_image = image

    else:
        json_output = {"text": raw_output}
        annotated_image = image

    return annotated_image, raw_output, json_output


with gr.Blocks(
    title="KhaṭṭVision",
    fill_width=True,
) as demo:
    gr.Markdown(
        """
# KhaṭṭVision — Arabic Calligraphy Understanding

Upload an Arabic calligraphy image and select a task.

The fine-tuned Muse Glimmer model can perform:

- Arabic calligraphy OCR
- Structured JSON analysis
- Calligraphy-style classification
- Theme classification
- Text-region localization
"""
    )

    with gr.Row():
        with gr.Column(scale=1):
            image_input = gr.Image(
                type="pil",
                label="Upload Arabic calligraphy",
                height=500,
            )

            task_input = gr.Dropdown(
                choices=list(TASKS.keys()),
                value="Structured analysis",
                label="Task",
            )

            run_button = gr.Button(
                "Analyze image",
                variant="primary",
            )

            clear_button = gr.ClearButton(
                value="Clear",
            )

        with gr.Column(scale=1):
            annotated_output = gr.Image(
                type="pil",
                label="Image and predicted regions",
                height=500,
            )

            with gr.Tab("Model response"):
                text_output = gr.Textbox(
                    label="Raw output",
                    lines=10,
                    show_copy_button=True,
                )

            with gr.Tab("Parsed result"):
                json_output = gr.JSON(
                    label="Structured output",
                )

    run_button.click(
        fn=run_khattvision,
        inputs=[image_input, task_input],
        outputs=[
            annotated_output,
            text_output,
            json_output,
        ],
        concurrency_limit=1,
    )

    clear_button.add(
        [
            image_input,
            annotated_output,
            text_output,
            json_output,
        ]
    )


demo.queue(default_concurrency_limit=1)

demo.launch(
    share=True,
    debug=False,
    show_error=True,
)